# Customer Segmentation with Clustering

K-Means and agglomerative clustering on 68,300 retail customers, with PCA and t-SNE for visualisation. See the README for known limitations.

*Origin: University of Cambridge Data Science Career Accelerator coursework (2024-25). Data source anonymised and not included. Cell outputs cleared to protect the source data; results quoted in the README come from the original run.*

### <font color="blue">**Following the Rubric Skills Criteria, the project has been divided into four main parts:**</font>

* <font color="blue">**PART I:** Initial data exploration and assessment
* **PART II:** Clustering with ML models
* **PART III:** Customer segments
* **PART IV:** Dimensionality reduction </font>

**Load the data**

 ### <font color="blue">**PART I:** Initial data exploration and assessment</font>

In [ ]:
# 1. Import the required libraries and data set with the provided URL

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.ensemble import IsolationForest

In [ ]:
df = pd.read_csv("data/customers.csv")  # place the data file in ./data
print(df.head())

In [ ]:
df.head()

In [ ]:
# Make a copy of the df
data = df.copy()
print(type(data))
print()
print()
data.describe()
print()
print()
data.info()
print()
print()
data.columns
print()
print()
data.shape
print()
print(f"Number of unique Customer ID values in the DataFrame: {data['Customer ID'].nunique()}")
print()
print()

In [ ]:
# Checking unique values for Continent and Country
print(data[['Continent', 'CustomerCountryLabel']].nunique())

In [ ]:
# Select only numeric columns
numeric_data = data.select_dtypes(include=[np.number])

# Identify rows with negative values in any numeric column (e.g. -$10)
negative_values = numeric_data[numeric_data < 0].dropna(how='all')

# Print or further analyse 'negative_values'
print(negative_values)

<font color="blue">**Missing and duplicate values in the data are identified**</font>

In [ ]:
# View the DataFrame and perform data pre-processing:
print('DataFrame first 5 rows:')
print()
print(data.head())
print()
print()
print('DataFrame information:')
print()
print(data.info())
print()
print()
print('DataFrame description:')
print()
print(data.describe())
print()
print()
print('DataFrame columns:')
print()
print(data.columns)
print()
print()

# 1. Identifying missing values
missing_values = data.isnull().sum()
print('Missing values:')
print()
print()
print(missing_values)
print()
print()

# 2. Identifying duplicated data
duplicated_data = data.duplicated().sum()
print('Duplicated data:')
print()
print(duplicated_data)
print()
print()

In [ ]:
# Dealing with missing data and duplicates

# Filling in missing values (NaNs) in the City, State_province, and Postal_code
# columns using the most frequent value (mode) of each respective column

data['City'] = data['City'].fillna(data['City'].mode()[0])
data['State_Province'] = data['State_Province'].fillna(data['State_Province'].mode()[0])
data['Postal_Code'] = data['Postal_Code'].fillna(data['Postal_Code'].mode()[0])

# removing duplicated data
data = data.drop_duplicates()

In [ ]:
# Checking data after handling missing and duplicated data

# Identifying missing values
missing_values = data.isnull().sum()
print('Missing values:')
print()
print()
print(missing_values)
print()
print()

# Identifying duplicated data
duplicated_data = data.duplicated().sum()
print('Duplicated data:')
print()
print(duplicated_data)
print()
print()

In [ ]:
# Get the data types of all columns
column_data_types = data.dtypes

# Filter to show only columns with 'object' data type
object_columns = column_data_types[column_data_types == 'object']

# Print the object columns and their data types
print(object_columns)

In [ ]:
print(data.nunique())

In [ ]:
# Get the data types of all columns
column_data_types = data.dtypes

# Filter to show only columns with 'object' data type
int64_columns = column_data_types[column_data_types == 'int64']

# Print the int64 columns and their data types
print(int64_columns)

<font color="blue">**Feature Engineering: Created new features for frequency, recency, CLV, average unit cost, and customer age (step 3)**</font>

<font color="blue">Here the code preprocesses data and adds columns (Frequency, Recency, CLV, Average_Unit_Cost, Customer_Age) for insights into customer behaviour.</font>

In [ ]:
# 3. Feature Engineering

# Function to clean and convert 'Unit Cost', 'Total Revenue' and 'Profit'
def clean_and_convert(df, column, dollar_sign=True):
    if dollar_sign:
        df[column] = df[column].str.replace('$', '', regex=False)
    df[column] = df[column].str.replace(',', '', regex=False)
    df[column] = df[column].str.replace(r'\((.*?)\)', r'-\1', regex=True)
    df[column] = pd.to_numeric(df[column], errors='coerce')

clean_and_convert(data, 'Unit Cost', dollar_sign=True)
clean_and_convert(data, 'Total Revenue', dollar_sign=True)
clean_and_convert(data, 'Profit', dollar_sign=True)

In [ ]:
# Calculate Customer Lifetime Value (CLV)

# First Calculate total revenue per customer
total_revenue_per_customer = data.groupby('Customer ID')['Total Revenue'].sum()

# Then Calculate total cost per customer (considering Unit Cost and Quantity)
total_cost_per_customer = (data['Unit Cost'] * data['Quantity']).groupby(data['Customer ID']).sum()

# Calculate CLV
clv = total_revenue_per_customer - total_cost_per_customer

# Display the CLV
print(clv)

In [ ]:
# Convert clv to a DataFrame for merging

clv_df = clv.reset_index()
clv_df.columns = ['Customer ID', 'CLV']
data = data.merge(clv_df, on='Customer ID', how='left')
print(clv_df.head())
print()
print()
print(data.head())
print()
print()
print(data.columns)
print()
print()
print(data.shape)
print(data.info())

In [ ]:
# Convert dates
data['Delivery_Date'] = pd.to_datetime(data['Delivery_Date'], format='%d%b%Y', errors='coerce')
data['Customer_BirthDate'] = pd.to_datetime(data['Customer_BirthDate'], format='%d%b%Y', errors='coerce')
data['Order_Date']= pd.to_datetime(data['Order_Date'], format='%d%b%Y', errors='coerce')

# Determining the time window for 'Delivery_Date' and 'Order_Date'

# Delivery_Date window
min_date = data['Delivery_Date'].min()
max_date = data['Delivery_Date'].max()
print(f"The delivery date range is from {min_date} to {max_date}")
print()
print()

# Order_Date window
min_order_date_data = data['Order_Date'].min()
max_order_date_data = data['Order_Date'].max()
print(f"The order date range is from {min_order_date_data} to {max_order_date_data}")

In [ ]:
# Given the time window determined above for Delivery and Order dates, I've
# set a fixed date for the analysis

# Determine the latest delivery date
fixed_analysis_date = data['Delivery_Date'].max()
print(f"Fixed analysis date set to: {fixed_analysis_date}")

# The analysis will be based on the most recent data available

In [ ]:
# Calculating additional features
data['Average_Unit_Cost'] = data.groupby('Customer ID')['Unit Cost'].transform('mean')
data['Frequency'] = data.groupby('Customer ID')['Order ID'].transform('count')

# Ensuring no negative frequency
data['Frequency'] = data['Frequency'].clip(lower=0) # ensuring no negative values

# Calculating Recency and Customer Age, handling NaNs
data['Recency'] = (fixed_analysis_date - data.groupby('Customer ID')['Delivery_Date'].transform('max')).dt.days.fillna(0) # tracks the number of days since the last delivery for each customer, with any missing values being filled
data['Customer_Age'] = ((fixed_analysis_date - data['Customer_BirthDate']).dt.days // 365).clip(lower=0) # ensuring no negative ages

In [ ]:
# Checking if there are missing values in 'Recency' and 'Customer_Age'

missing_recency = data['Recency'].isnull().sum()
missing_customer_age = data['Customer_Age'].isnull().sum()

print(f"Missing values in Recency: {missing_recency}")
print(f"Missing values in Customer Age: {missing_customer_age}")

<font color="blue">The **data_aggr** DataFrame has one row per customer, with aggregated values for Average_Unit_Cost, Frequency, CVL, Recency, and Customer_Age (listed as steps 2 & 3 - **PART I**).</font>

In [ ]:
# Making a copy of the data set
data_selected = data.copy()

# Here the code performs data aggregation to create a summarised DataFrame where
# each row represents a unique customer.
# Defining aggregation operations for each Engineered Feature
agg_operations = {
    'Average_Unit_Cost': 'mean',
    'Frequency': 'max',
    'CLV': 'sum',
    'Recency': 'min',
    'Customer_Age': 'mean'
}

# Data set with one customer per row
data_aggr = data_selected.groupby('Customer ID').agg(agg_operations).reset_index()

In [ ]:
# Check for missing data in data_aggr
missing_data = data_aggr.isnull().sum()
print(missing_data)

In [ ]:
print(data_aggr.head())
print()
print()
print(data_aggr.columns)
print()
print()
print(data_aggr.shape)
print()
print()
print(data_aggr.info())
print()
print()
print(data_aggr.describe())
print()
print()
print(data_aggr.nunique())

<font color="blue">**Determining if there are any outliers** (listed as step 2)</font>

In [ ]:
# 2. Identifying Outliers using boxplot using data set BEFORE normalising

data_aggr_IQR_cal = data_aggr.drop('Customer ID', axis=1)

plt.figure(figsize=(12, 8))
sns.boxplot(data=data_aggr_IQR_cal, palette='Set2')
plt.title('Boxplot of data_agg before standardising data', fontsize=14)
plt.xlabel('Features', fontsize=12)
plt.ylabel('Values', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

**Analysis**

* CLV: High variability, wide range, many outliers
* Other features: Stable with less variability
* Outliers: Especially in CLV, require cleaning for robust analysis
* Skewness: Right-skewed CLV, influenced by few high-value customers

In [ ]:
# Function to calculate IQR Outliers before standardising

def iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    lower_bound = Q1 - 1.5 * IQR
    is_outlier = (series < lower_bound) | (series > upper_bound)
    return is_outlier, Q1, Q3, upper_bound, lower_bound

# Apply the IQR outlier detection method to each feature
outlier_counts = {}
for column in data_aggr_IQR_cal.columns:
    is_outlier, Q1, Q3, upper_bound, lower_bound = iqr_outliers(data_aggr_IQR_cal[column])
    outlier_counts[column] = is_outlier.sum()
    print(f'For {column}:')
    print(f'Q1: {Q1}')
    print(f'Q3: {Q3}')
    print(f'Number of outliers: {outlier_counts[column]}')
    print()
    print(f"upper_bound: {upper_bound}")
    print(f"lower_bound: {lower_bound}")
    print()
    print()

    # Create new column for outlier flag
    new_column_name = f"{column}_is_outlier"
    data_aggr_IQR_cal[new_column_name] = is_outlier.astype(int)

# Print the total number of outliers for each feature
for column, count in outlier_counts.items():
    print(f'{column}: {count} outliers')
    print()

# Visualisation of outliers for each feature
plt.figure(figsize=(15, 10))
for i, column in enumerate(data_aggr_IQR_cal.columns):
    if 'outlier' not in column:
        # Check if the column has enough data for a boxplot
        if len(data_aggr_IQR_cal[column].dropna()) > 0:  # Check if column is not empty after removing NaNs
            plt.subplot((len(data_aggr_IQR_cal.columns) + 1) // 2, 2, i + 1)
            sns.boxplot(x=data_aggr_IQR_cal[column])
            plt.title(f'{column}')
        else:
            print(f"Skipping boxplot for '{column}' due to insufficient data.")
plt.tight_layout()
plt.show()

In [ ]:
print(data_aggr_IQR_cal.columns)

**Analysis**

* Unit Cost: Low variability, median near zero, few outliers
* Frequency: Highly right-skewed, few very frequent customers
* CLV: Wide range, significant outliers, high-value focus
* Recency: Broad IQR, clustered data, varied engagement
* Age: Normal, concentrated middle, typical demographic

In [ ]:
# Plot histograms for each column in data_aggr_IQR_cal
cols_to_plot = [col for col in data_aggr_IQR_cal.columns if 'is_outlier' not in col]
data_aggr_IQR_cal[cols_to_plot].hist(bins=20, figsize=(15, 10), grid=True)

# Add a title
plt.suptitle('Histograms of data_aggr_IQR_cal', fontsize=16, y=1.10)

plt.tight_layout()

# Display the plots
plt.show()

**Distribution**

* Unit Cost: Concentrated low, few high-cost outliers.
* Frequency: Right-skewed, low engagement for most.
* CLV: Mostly low, with rare high-value customers.
* Recency: Varied engagement times, spread distribution.
* Age: Normal distribution, 30/35-50 age range.

In [ ]:
# Defining again function to find outliers based on IQR only

# Function to calculate IQR outliers
def iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    is_outlier = (series < lower_bound) | (series > upper_bound)
    return is_outlier, Q1, Q3

# Apply the IQR outlier detection method to each feature
outlier_counts = {}
for column in data_aggr_IQR_cal.columns:
    if 'outlier' not in column:  # Avoid processing the outlier columns again
        data_aggr_IQR_cal[column + '_is_outlier'], Q1, Q3 = iqr_outliers(data_aggr_IQR_cal[column])
        outlier_counts[column] = data_aggr_IQR_cal[column + '_is_outlier'].sum()
        print(f'For {column}:')
        print(f'Q1: {Q1}')
        print(f'Q3: {Q3}')
        print(f'Number of outliers: {outlier_counts[column]}')
        print()

In [ ]:
print(data_aggr_IQR_cal.columns)

In [ ]:
# Determinig Outliers with Isolation Forest

data_aggr_IF_cal = data_aggr_IQR_cal # data_aggr.drop('Customer ID', axis=1)
data_scaled_df = data_aggr_IF_cal.copy()  # Create a copy of data_aggr_IF_cal

# features for fitting the model
X = data_aggr_IF_cal[['Average_Unit_Cost', 'Frequency', 'CLV', 'Recency', 'Customer_Age']]
print(X.head())
print()
print()

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the Isolation Forest model
model = IsolationForest(contamination=0.05, random_state=42) # setting to 5% outliers
model.fit(X_scaled)
y_pred = model.predict(X_scaled)
data_scaled_df['Outlier'] = y_pred

# Print the value counts of predictions (-1 for outliers, 1 for inliers)
print(data_scaled_df['Outlier'].value_counts())

In [ ]:
# Anomalies = Outliers - obtained using the Isolation Forest Method ~ 5%

# Display outliers in the DataFrame.
anomalies = data_scaled_df[data_scaled_df.Outlier == -1]

# View output.
print(anomalies.shape)
print()
print(f"Number of anomalies inthis calculation indicates it is around 5% of the total sample: {anomalies.shape[0]}")
print()
print()
print(anomalies)

In [ ]:
np.random.seed(0)  # For reproducibility

def identify_outliers_by_feature_count(data, n_features_required):
    outlier_columns = [col for col in data.columns if col.endswith('_is_outlier')]
    print(f'Outlier columns: {outlier_columns}')  # Debug Statement
    data_copy = data.copy()
    data_copy['total_outliers'] = data_copy[outlier_columns].sum(axis=1)
    observations_with_n_outliers = data_copy[data_copy['total_outliers'] >= n_features_required]
    total_samples = len(data_copy)
    outlier_percentage = (len(observations_with_n_outliers) / total_samples) * 100
    return observations_with_n_outliers, outlier_percentage

# Identify outliers with number of features flagged
for n in range(1, len(data_scaled_df.columns) - len(data_scaled_df.columns) // 2 + 1):
    obs_with_n_outliers, percentage = identify_outliers_by_feature_count(data_scaled_df, n)
    print(f'Number of features required: {n}, Outlier Percentage: {percentage:.2f}%, Number of Outliers: {len(obs_with_n_outliers)}')

# Loop through to find the number of features that results in 1-5% of samples being outliers
for num_features_required in range(1, len(data_scaled_df.columns) - len(data_scaled_df.columns) // 2 + 1):
    obs_with_n_outliers, outlier_percentage = identify_outliers_by_feature_count(data_scaled_df, num_features_required)

    if 1 <= outlier_percentage <= 5:
        print(f"Number of features that must simultaneously be in outlier condition for a sample to be classified as an outlier: {num_features_required}")
        print(f"Percentage of samples classified as outliers: {outlier_percentage:.2f}%")
        print(f"Number of outlier samples: {len(obs_with_n_outliers)}")
        print()
        break
else:
    print("No condition found where 1-5% of samples are outliers.")
    print()

In [ ]:
print(data_scaled_df.columns)
print()
print()
print(data_scaled_df.shape)
print()

In [ ]:
# Recalculating by using the flags only
data_recal = data_scaled_df.copy()
data_recal = data_recal[['Average_Unit_Cost_is_outlier', 'Frequency_is_outlier', 'CLV_is_outlier', 'Recency_is_outlier', 'Customer_Age_is_outlier']]
data_recal.columns

# Sum the flagged columns for each observation
outliers_recalculated = (data_recal == 1).sum(axis=1)
print()
outliers_recalculated.describe()
print()
outliers_recalculated.info()

In [ ]:
# Create a mask for observations with at least two flagged columns
mask = outliers_recalculated >= 2

# Filter the DataFrame to get observations that meet the criteria
observations_with_at_least_two_flags = data_recal[mask]

total_true_observations = mask.sum()

# Print the result
print(f'Total number of observations where mask is True: {total_true_observations}')

<font color="blue">**Given the non-gaussian distribution of the data, opted to remove the outliers obtained with the Isolation Forest Method**</font>

In [ ]:
# Removing Outliers (5%)

data_clean = data_scaled_df[~data_scaled_df['Outlier'].isin([-1])] # removes the rows that have been identified as outliers by the Isolation Forest model.
data_aggr_IQR_cal_cleaned = data_clean.copy()
data_aggr_IQR_cal_cleaned = data_aggr_IQR_cal_cleaned.drop(['Average_Unit_Cost_is_outlier', 'Frequency_is_outlier', 'CLV_is_outlier', 'Recency_is_outlier', 'Customer_Age_is_outlier', 'Outlier'], axis=1)
print(data_clean.shape)
print()
print()
print(data_clean.head())
print()
print()
print(data_clean.describe())
print()
print()
print(data_clean.columns)
print()
print()
print(data_clean.shape)
print()
print()
print(data_clean.nunique())

### <font color="blue">**PART II:** Clustering with ML models</font>

In [ ]:
# Function to find outliers using IQR method
def find_outliers(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    iqr = Q3 - Q1
    lower_bound = Q1 - 1.5 * iqr
    upper_bound = Q3 + 1.5 * iqr
    outliers = (column < lower_bound) | (column > upper_bound)
    # It returns these statistics along with a boolean indicator for whether any outliers exist
    # Check if outliers exist
    any_outliers = (outliers == 1).any()
    return Q1, Q3, lower_bound, upper_bound, any_outliers

# Function to make histograms for each column
def plot_histograms(dataframe, columns):
    # Filter out columns that say 'is_outlier'
    columns_to_plot = []
    for col in columns:
        if 'is_outlier' not in col and col != 'Outlier':
            columns_to_plot.append(col)

    if len(columns_to_plot) == 0:
        print("No columns left to plot after removing 'is_outlier' ones.")
        return

    # Create a new figure for all plots
    plt.figure(figsize=(20, 5*len(columns_to_plot)))

    for i, col in enumerate(columns_to_plot):
        plt.subplot(len(columns_to_plot), 1, i + 1)

        # Histogram
        sns.histplot(dataframe[col], kde=True, bins=10, stat='density',
                     edgecolor='black', color='#1f77b4')

        plt.title(f'Histogram for {col}')
        plt.xlabel(col)
        plt.ylabel('Density')

        # Statistics
        Q1, Q3, lower_bound, upper_bound, _ = find_outliers(dataframe[col])
        mean_value = dataframe[col].mean()

        # Lines for stats
        plt.axvline(Q1, color='black', linestyle='--', label='Q1')
        plt.axvline(Q3, color='red', linestyle='--', label='Q3')
        plt.axvline(lower_bound, color='blue', linestyle='--', label='Lower Bound')
        plt.axvline(upper_bound, color='green', linestyle='--', label='Upper Bound')
        plt.axvline(mean_value, color='purple', linestyle='--', label='Mean')

        # Labels for each line
        plt.text(Q1, plt.ylim()[1]*0.9, f'Q1: {Q1:.2f}', rotation=90, va='top', ha='right', color='black')
        plt.text(Q3, plt.ylim()[1]*0.9, f'Q3: {Q3:.2f}', rotation=90, va='top', ha='right', color='red')
        plt.text(lower_bound, plt.ylim()[1]*0.9, f'Lower: {lower_bound:.2f}', rotation=90, va='top', ha='right', color='blue')
        plt.text(upper_bound, plt.ylim()[1]*0.9, f'Upper: {upper_bound:.2f}', rotation=90, va='top', ha='right', color='green')
        plt.text(mean_value, plt.ylim()[1]*0.9, f'Mean: {mean_value:.2f}', rotation=90, va='top', ha='right', color='purple')

        plt.legend()
        plt.grid(True)

    plt.tight_layout()
    plt.show()

<font color="blue">**Visualisations to explore the data BEFORE removing outliers**</font>

In [ ]:
# 4. Perform EDA and create visualisations to explore the data BEFORE removing
# outliers.
plot_histograms(data_aggr_IQR_cal, data_aggr_IQR_cal.columns)

<font color="blue">**Visualisations (below) to explore the data AFTER removing outliers**</font>

In [ ]:
# 4. Perform EDA and create visualisations to explore the data AFTER removing
# outliers (outliers were identified using Isolation Forest Method). data_clean
plot_histograms(data_clean, data_clean.columns)

<font color="blue">**Observing Negative lower bound results for some features  likely due to issues with data integrity or non- gaussian distribution - The code below is recalculating lower bounds to ensure positive results**</font>

In [ ]:
# Function to find outliers using IQR method
def find_outliers2(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    iqr = Q3 - Q1
    lower_bound = max(Q1 - 1.5 * iqr, 0)
    upper_bound = Q3 + 1.5 * iqr # To ensure positive results
    outliers = (column < lower_bound) | (column > upper_bound)

    # Check if outliers exist
    any_outliers = (outliers == 1).any()

    return Q1, Q3, lower_bound, upper_bound, any_outliers

# Histograms for each column
def plot_histograms2(dataframe, columns):
   # Filter out columns that say 'is_outlier' and 'Outlier'
    columns_to_plot = []
    for col in columns:
        if 'is_outlier' not in col and col != 'Outlier':
            columns_to_plot.append(col)

    if len(columns_to_plot) == 0:
        print("No columns left to plot after removing 'is_outlier' ones.")
        return

    # New figure for all plots
    plt.figure(figsize=(20, 5*len(columns_to_plot)))

    for i, col in enumerate(columns_to_plot):
        plt.subplot(len(columns_to_plot), 1, i + 1)

        # histogram
        sns.histplot(dataframe[col], kde=True, bins=10, stat='density',
                     edgecolor='black', color='#1f77b4')

        plt.title(f'Histogram for {col}')
        plt.xlabel(col)
        plt.ylabel('Density')

        # Statistics
        Q1, Q3, lower_bound, upper_bound, _ = find_outliers2(dataframe[col])
        mean_value = dataframe[col].mean()

        # Lines for statistics
        plt.axvline(Q1, color='black', linestyle='--', label='Q1')
        plt.axvline(Q3, color='red', linestyle='--', label='Q3')
        plt.axvline(lower_bound, color='blue', linestyle='--', label='Lower Bound')
        plt.axvline(upper_bound, color='green', linestyle='--', label='Upper Bound')
        plt.axvline(mean_value, color='purple', linestyle='--', label='Mean')

        # Labels
        plt.text(Q1, plt.ylim()[1]*0.9, f'Q1: {Q1:.2f}', rotation=90, va='top', ha='right', color='black')
        plt.text(Q3, plt.ylim()[1]*0.9, f'Q3: {Q3:.2f}', rotation=90, va='top', ha='right', color='red')
        plt.text(lower_bound, plt.ylim()[1]*0.9, f'Lower: {lower_bound:.2f}', rotation=90, va='top', ha='right', color='blue')
        plt.text(upper_bound, plt.ylim()[1]*0.9, f'Upper: {upper_bound:.2f}', rotation=90, va='top', ha='right', color='green')
        plt.text(mean_value, plt.ylim()[1]*0.9, f'Mean: {mean_value:.2f}', rotation=90, va='top', ha='right', color='purple')

        plt.legend()
        plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# 4. Perform EDA and create visualisations to explore the data AFTER removing
# outliers and encuring positive values for lower bounds
plot_histograms2(data_clean, data_clean.columns)

**Analysis after recalculating to ensure no negative lower bounds**

* Unit Cost: Lower, more consistent values
* Frequency: Low peak, right-skewed, tighter distribution
* CLV: Peaks low, no negatives, long high-value tail
* Recency: Peaks at 0, less spread, more consistent
* Age: closer to a normal distribution

In [ ]:
print(data_clean.describe())

In [ ]:
# 5. For more efficient programming, incorporate column transformer
 # (and pipeline)

# Standardising the clean data
scaler = StandardScaler()
data_clean_scaled = scaler.fit_transform(data_clean)
data_clean_scaled = pd.DataFrame(data_clean_scaled, columns=data_clean.columns)

# Display the DataFrame to verify
print(data_clean_scaled.head())
print()
print()
print(data_clean_scaled.describe())

In [ ]:
data_clean_scaled = data_clean_scaled.drop(['Average_Unit_Cost_is_outlier', 'Frequency_is_outlier', 'CLV_is_outlier', 'Recency_is_outlier', 'Customer_Age_is_outlier', 'Outlier'], axis = 1)
print(data_clean_scaled.columns)

In [ ]:
sse = [] # This list will store the sum of squared errors for each value of k.

# Loop to try different numbers of clusters
for k in range(1, 11):
    # KMeans model
    kmeans = KMeans(n_clusters=k, random_state=42)
    # Fit the model to data
    kmeans.fit(data_clean_scaled)
    # SSE
    sse.append(kmeans.inertia_)
    print(f'For k = {k}, SSE = {kmeans.inertia_}')

# The "elbow point" where the SSE starts to level off
# indicating the optimal number of clusters
plt.figure(figsize=(10, 5))
# SSE for each number of clusters
plt.plot(range(1, 11), sse, marker='o')
# Set x-axis ticks to number from 1 to 10
plt.xticks(range(1, 11))
plt.xlabel('Number of Clusters')
plt.ylabel('SSE')
plt.title('Elbow Method for Optimal Number of Clusters')
plt.grid(True)
plt.show()

**Elbow Method Notes**

 From visual inspection best k appears to be 5. After five the decrease in SSE appears more gradual.

In [ ]:
# Silhouette method

# List to store silhouette scores
silhouette_scores = []

for k in range(2, 11):
    # Create KMeans model
    kmeans = KMeans(n_clusters=k, random_state=42)
    # Fit and predict
    cluster_labels = kmeans.fit_predict(data_clean_scaled)
    # silhouette score
    silhouette_avg = silhouette_score(data_clean_scaled, cluster_labels)
    # average silhouette score
    silhouette_scores.append(silhouette_avg)
    print(f'For k = {k}, Silhouette Score = {silhouette_avg}')

# silhouette method plotting
plt.figure(figsize=(10, 5))
# Plot the silhouette scores
plt.plot(range(2, 11), silhouette_scores, marker='o')
# Set x-axis ticks to show each number from 2 to 10
plt.xticks(range(2, 11))
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Method for Optimal Number of Clusters')
plt.grid(True)
plt.show()

**Silhouette Method Notes:**

The best number of clusters (k) would be the one with the highest Silhouette Score(s): 2 and/or 5.

For the calculations below will attempt to check both values and determine which is best to indicate well-defined clusters.

In [ ]:
# Number of clusters, here I use 4 based on the
# Elbow Method
kmeans = KMeans(n_clusters=4, random_state=42)

# Fit KMeans model to scaled data
kmeans.fit(data_clean_scaled)

# Predict clusters for data
cluster_labels = kmeans.predict(data_clean_scaled)

# Show the cluster labels
print(cluster_labels)

In [ ]:
# Number of clusters, here I use 2 based on the
# Silhouette Method
kmeans = KMeans(n_clusters=2, random_state=42)

# Fit KMeans model to scaled data
kmeans.fit(data_clean_scaled)

# Predict clusters for data
cluster_labels = kmeans.predict(data_clean_scaled)

# Show the cluster labels
print(cluster_labels)

In [ ]:
# Perform k-means clustering using scikit-learn - using a conservative n_cluster = 4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
data_clean_scaled['cluster'] = kmeans.fit_predict(data_clean_scaled[['CLV', 'Recency']])

# Centroids
centroids = kmeans.cluster_centers_

# Plotting the clusters and centroids
for cluster_idx in range(4):
    cluster_data = data_clean_scaled[data_clean_scaled['cluster'] == cluster_idx]
    plt.scatter(cluster_data['CLV'], cluster_data['Recency'], label=f'Cluster {cluster_idx + 1}')
    plt.scatter(centroids[cluster_idx][0], centroids[cluster_idx][1], s=200, marker='X', edgecolor='black', color='red')

plt.xlabel('CLV')
plt.ylabel('Recency')
plt.legend()
plt.title('K-Means (4) Clustering: Recency vs CLV')
plt.show()

# Print final centroids
print("Final Centroids - n_clusters = 4:", "\n", centroids)

**Cluster Analysis**

Recency vs. CLV:
* 1: Recent, low-value
* 2: Very recent, low contribution
* 3: High-value, less recent
* 4: Moderate value, average recency

In [ ]:
# Perform k-means clustering using scikit-learn - using 5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
data_clean_scaled['cluster'] = kmeans.fit_predict(data_clean_scaled[['CLV', 'Recency']])

# Centroids
centroids = kmeans.cluster_centers_

# Plotting the clusters and centroids
for cluster_idx in range(5):
    cluster_data = data_clean_scaled[data_clean_scaled['cluster'] == cluster_idx]
    plt.scatter(cluster_data['CLV'], cluster_data['Recency'], label=f'Cluster {cluster_idx + 1}')
    plt.scatter(centroids[cluster_idx][0], centroids[cluster_idx][1], s=200, marker='X', edgecolor='black', color='red')

plt.xlabel('CLV')
plt.ylabel('Recency')
plt.legend()
plt.title('K-Means (5) Clustering: Recency vs CLV')
plt.show()

# Print final centroids
print("Final Centroids - n_clusters = 5:", "\n", centroids)

**CLuster Analysis**

* 1 (Blue): Recent, low CLV.
* 2 (Orange): Recent, negative CLV.
* 3 (Green): Moderate CLV & recency.
* 4 (Red): High CLV, less recent.
* 5 (Purple): Very recent, very negative CLV.

In [ ]:
# Perform k-means clustering using scikit-learn - using 2
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
data_clean_scaled['cluster'] = kmeans.fit_predict(data_clean_scaled[['CLV', 'Recency']])

# centroids
centroids = kmeans.cluster_centers_

# Plotting the clusters and centroids
for cluster_idx in range(2):
    cluster_data = data_clean_scaled[data_clean_scaled['cluster'] == cluster_idx]
    plt.scatter(cluster_data['CLV'], cluster_data['Recency'], label=f'Cluster {cluster_idx + 1}')
    plt.scatter(centroids[cluster_idx][0], centroids[cluster_idx][1], s=200, marker='X', edgecolor='black', color='red')

plt.xlabel('CLV')
plt.ylabel('Recency')
plt.legend()
plt.title('K-Means (2) Clustering: Recency vs CLV')
plt.show()

# Print final centroids
print("Final Centroids - n_clusters = 2:", "\n", centroids)

**Cluster Analysis**

* 1 (Blue): Recent, low CLV
* 2 (Orange): Valuable, less recent

In [ ]:
# Perform k-means clustering using scikit-learn - usigng 4
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
data_clean_scaled['cluster'] = kmeans.fit_predict(data_clean_scaled[['CLV', 'Customer_Age']])

# centroids
centroids = kmeans.cluster_centers_

# Plotting the clusters and centroids
for cluster_idx in range(5):
    cluster_data = data_clean_scaled[data_clean_scaled['cluster'] == cluster_idx]
    plt.scatter(cluster_data['CLV'], cluster_data['Customer_Age'], label=f'Cluster {cluster_idx + 1}')
    plt.scatter(centroids[cluster_idx][0], centroids[cluster_idx][1], s=200, marker='X', edgecolor='black', color='red')

plt.xlabel('CLV')
plt.ylabel('Customer_Age')
plt.legend()
plt.title('K-Means (5) Clustering: Customer_Age vs CLV')
plt.show()

# Print final centroids
print("Final Centroids - n_clusters = 5:", "\n", centroids)

**Cluster Analysis**

* 1 (Blue): Young, negative CLV
* 2 (Orange): Young, positive CLV
* 3 (Green): Old, negative CLV
* 4 (Red): Broad age, high CLV
* 5 (Purple): Mixed ages, average CLV

In [ ]:
# Perform k-means clustering using scikit-learn - using 2
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
data_clean_scaled['cluster'] = kmeans.fit_predict(data_clean_scaled[['CLV', 'Customer_Age']])

# centroids
centroids = kmeans.cluster_centers_

# Plotting the clusters and centroids
for cluster_idx in range(2):
    cluster_data = data_clean_scaled[data_clean_scaled['cluster'] == cluster_idx]
    plt.scatter(cluster_data['CLV'], cluster_data['Customer_Age'], label=f'Cluster {cluster_idx + 1}')
    plt.scatter(centroids[cluster_idx][0], centroids[cluster_idx][1], s=200, marker='X', edgecolor='black', color='red')

plt.xlabel('CLV')
plt.ylabel('Customer_Age')
plt.legend()
plt.title('K-Means (2) Clustering: Customer Age vs CLV')
plt.show()

# Print final centroids
print("Final Centroids - n_clusters = 2:", "\n", centroids)

**Cluster Analysis**

* 1 (Blue): Older, lower CLV
* 2 (Orange): Younger, higher CLV

In [ ]:
# 8. Based on the optimum number of k (both 2 and 5), perform k-means clustering

data_2 = data_clean_scaled.copy()

# Perform K-means clustering
kmeans = KMeans(n_clusters=5,
                random_state=42,
                n_init=10)
data_2['cluster'] = kmeans.fit_predict(data_2[['Customer_Age', 'CLV']])

# centroids
centroids = kmeans.cluster_centers_

# Plotting clusters and centroids
for cluster_idx in range(5):
    cluster_data = data_2[data_2['cluster'] == cluster_idx]
    plt.scatter(cluster_data['Customer_Age'],
                cluster_data['CLV'],
                label=f'Cluster {cluster_idx + 1}')
    plt.scatter(*centroids[cluster_idx],
                s=200, marker='X',
                edgecolor='black')

plt.xlabel('Customer_Age')
plt.ylabel('CLV')
# plt.xlim(0, 2)
# plt.ylim(-2.5, 2.5)
plt.legend()
plt.show()

# Printing final centroids
print("Final Centroids - n_clusters = 5:", "\n", centroids)

**Cluster Analysis**

* 1 (Blue): Low CLV, young
* 2 (Green): Moderate CLV, old
* 3 (Pink): High CLV, young
* 4 (Purple): Low CLV, very old
* 5 (Yellow): Balanced CLV, mixed ages

In [ ]:
# 8. Based on the optimum number of k (both 2 and 5), perform k-means clustering

data_3 = data_clean_scaled.copy()

# Perform K-means clustering
kmeans = KMeans(n_clusters=2,
                random_state=42,
                n_init=10)
data_3['cluster'] = kmeans.fit_predict(data_3[['Customer_Age', 'CLV']])

# centroids
centroids = kmeans.cluster_centers_

# Plotting clusters and centroids.
for cluster_idx in range(2):
    cluster_data = data_3[data_3['cluster'] == cluster_idx]
    plt.scatter(cluster_data['Customer_Age'],
                cluster_data['CLV'],
                label=f'Cluster {cluster_idx + 1}')
    plt.scatter(*centroids[cluster_idx],
                s=200, marker='X',
                edgecolor='black')

plt.xlabel('Customer_Age')
plt.ylabel('CLV')
# plt.xlim(0, 2)
# plt.ylim(-2.5, 2.5)
plt.legend()
plt.show()

# Print final centroids
print("Final Centroids - n_clusters = 2:", "\n", centroids)

**Cluster Analysis**

* 1 (Green): High CLV, young
* 2 (Blue): Low CLV, old

In [ ]:
# Function to perform k-means clustering and plot the clusters in 2D - comparing
# unscaled data vs scaled data 2D visualisations - n_clusters = 2

def plot_k_means(data, features, scaled=False):
    # Extract the selected features
    X = data_clean[features].values

    # Optionally scale the features
    if scaled:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

    # Perform k-means clustering. Trying with k=2 first
    kmeans = KMeans(n_clusters=2, random_state=42,  n_init=10)
    kmeans.fit(X)
    labels = kmeans.labels_

    # 2D scatter plot
    plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis')

    # Plotting centroids.
    centroids = kmeans.cluster_centers_
    plt.scatter(centroids[:, 0],
                centroids[:, 1],
                s=100, c='red',
                marker='X')

    # Labels
    plt.xlabel(features[0])
    plt.ylabel(features[1])

    # Title
    if scaled:
        plt.title('K-means (n_clusters = 2) clustering with scaled features (2D)')
    else:
        plt.title('K-means (n_clusters = 2) clustering with unscaled features (2D)')

    # Show plot.
    plt.show()

# Choose two features for 2D plotting.
selected_features = ['Customer_Age', 'CLV']

# Plot unscaled data.
plot_k_means(data_clean, selected_features, scaled=False)

# Plot scaled data.
plot_k_means(data_clean, selected_features, scaled=True)

In [ ]:
# Function to perform k-means clustering and plot the clusters in 2D -
# n_clusters = 5

def plot_k_means(data, features, scaled=False):
    # Extract the selected features
    X = data_clean[features].values

    # Optionally scale the features
    if scaled:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

    # Perform k-means clustering. Trying with k=4 first
    kmeans = KMeans(n_clusters=5, random_state=42,  n_init=10)
    kmeans.fit(X)
    labels = kmeans.labels_

    # 2D scatter plot.
    plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis')

    # Plotting centroids.
    centroids = kmeans.cluster_centers_
    plt.scatter(centroids[:, 0],
                centroids[:, 1],
                s=100, c='red',
                marker='X')

    # Labeling axes.
    plt.xlabel(features[0])
    plt.ylabel(features[1])

    # Title.
    if scaled:
        plt.title('K-means (n_clusters = 5) clustering with scaled features (2D)')
    else:
        plt.title('K-means (n_clusters = 5) clustering with unscaled features (2D)')

    # Show plot.
    plt.show()

# Choose two features for 2D plotting.
selected_features = ['Customer_Age', 'CLV']

# Plot unscaled data.
plot_k_means(data_clean, selected_features, scaled=False)

# Plot scaled data.
plot_k_means(data_clean, selected_features, scaled=True)

**Analysis - Scaled Plot**

* 1 (Purple): Young, low CLV.
* 2 (Blue): Varied ages, moderate CLV.
* 3 (Green): Older, high CLV.
* 4 (Teal): Older, lower CLV.
* 5 (Yellow): Middle-aged, diverse CLV.

In [ ]:
# Checking data_clean_scaled columns

print(data_clean_scaled.columns)
data_clean_scaled2 = data_clean_scaled.copy()
data_clean_scaled.drop('cluster', axis=1, inplace=True)
print(data_clean_scaled.columns)

In [ ]:
# 7. Perform hierarchical clustering and create a dendogram using
# data_clean_scaled. Here k = 2 - CLV and Customer_Age

small_df = data_clean_scaled[['Customer_Age', 'CLV']].sample(n=10000).reset_index(drop=True)

# Hierarchical clustering from scratch
def hierarchical_clustering(data, method='average'):
    dists = pdist(data, metric='euclidean')
    linkage_matrix = linkage(dists, method=method)
    return linkage_matrix

# Compute the linkage matrix
linkage_matrix = hierarchical_clustering(small_df)

# Number of clusters we want to extract
k = 2

# Retrieve the clusters
clusters = fcluster(linkage_matrix, k, criterion='maxclust')

# Plot the data and colour by cluster
plt.scatter(small_df['Customer_Age'], small_df['CLV'], c=clusters, cmap='viridis')
plt.title('Hierarchical clustering of CLV and Customer_Age')
plt.xlabel('Customer_Age')
plt.ylabel('CLV')
# plt.xlim(-2.5, 2.5)
# plt.ylim(-2, 7)
plt.colorbar(label='Cluster')
plt.show()

# Plot the dendrogram
plt.figure(figsize=(35, 10))
dendrogram(linkage_matrix)
plt.title('Dendrogram')
plt.tick_params(labelbottom=False)
plt.xlabel('Sample index or (Cluster size)')
plt.ylabel('Distance')
plt.show()

**Cluster Analysis**

* 1 (Yellow): Lower CLV, younger skew.
* 2 (Purple): Higher CLV, older, long-term value
* Younger = lower CLV, possibly new or less engaged
* Older = higher CLV, loyalty

**Dendrogram:**
* Two main branches for two clusters.

In [ ]:
# 7. Perform hierarchical clustering and create a dendogram using
# data_clean_scaled. Here k = 5 - CLV and Customer_Age

small_df = data_clean_scaled[['Customer_Age', 'CLV']].sample(n=10000).reset_index(drop=True)

# Hierarchical clustering function
def hierarchical_clustering(data, method='ward'):
    dists = pdist(data, metric='euclidean')
    linkage_matrix = linkage(dists, method=method)
    return linkage_matrix

# Compute the linkage matrix
linkage_matrix = hierarchical_clustering(small_df)

# Number of clusters we want to extract
k = 5

# Retrieve the clusters
clusters = fcluster(linkage_matrix, k, criterion='maxclust')

# Plot the data and colour by cluster
plt.scatter(small_df['Customer_Age'], small_df['CLV'], c=clusters, cmap='viridis')
plt.title('Hierarchical clustering of CLV and Customer_Age')
plt.xlabel('Customer_Age')
plt.ylabel('CLV')
plt.xlim(small_df['Customer_Age'].min(), small_df['Customer_Age'].max())
plt.ylim(small_df['CLV'].min(), small_df['CLV'].max())
plt.colorbar(label='Cluster')
plt.show()

# Plot the dendrogram
plt.figure(figsize=(25, 10))
dendrogram(linkage_matrix)
plt.tick_params(labelbottom=False)
plt.title('Dendrogram')
plt.xlabel('Sample index or (Cluster size)')
plt.ylabel('Distance')
plt.show()

**Cluster Analysis**

* Segmentation by age and CLV
* Purple clusters possibly older, loyal customers
* Teal clusters low CLV, newer or less engaged
* The Dendrogram is assisting in determining optimal cluster number

In [ ]:
small_df.describe()

In [ ]:
# 7. Perform hierarchical clustering and create a dendogram using
# data_clean_scaled. Here k = 5 - Recency and CLV

small_df = data_clean_scaled[['Recency', 'CLV']].sample(n=10000).reset_index(drop=True)

# Hierarchical clustering from scratch
def hierarchical_clustering(data, method='average'):
    dists = pdist(data, metric='euclidean')
    linkage_matrix = linkage(dists, method=method)
    return linkage_matrix

# Compute the linkage matrix
linkage_matrix = hierarchical_clustering(small_df)

# Number of clusters we want to extract
k = 5

# Retrieve the clusters
clusters = fcluster(linkage_matrix, k, criterion='maxclust')

# Plot the data and colour by cluster
plt.scatter(small_df['Recency'], small_df['CLV'], c=clusters, cmap='viridis')
plt.title('Hierarchical clustering of Recency and CLV')
plt.xlabel('Recency')
plt.ylabel('CLV')
# plt.xlim(-1, 4)
# plt.ylim(-1, 7)
plt.colorbar(label='Cluster')
plt.show()

# Plot the dendrogram
plt.figure(figsize=(35, 10))
plt.tick_params(labelbottom=False)
dendrogram(linkage_matrix)
plt.title('Dendrogram')
plt.xlabel('Sample index or (Cluster size)')
plt.ylabel('Distance')
plt.show()

**Cluster Analysis**

* 5 (Yellow): High CLV, low recency; recent, high-value
* 4 (Green): High CLV, mixed recency
* 3 (Teal): Lower CLV, moderate recency
* 2 (Blue): Lower CLV, high recency; less recent, low-value
* 1 (Purple): Very low/negative CLV; inactive/lost

**Dendrogram:** Two main branches

In [ ]:
small_df = data_clean_scaled.sample(n=10000).reset_index(drop=True)

# Hierarchical clustering function
def hierarchical_clustering(data, method='ward'):
    dists = pdist(data, metric='euclidean')
    linkage_matrix = linkage(dists, method=method)
    return linkage_matrix

# Compute the linkage matrix
linkage_matrix = hierarchical_clustering(small_df)

# Number of clusters we want to extract
k = 5

# Retrieve the clusters
clusters = fcluster(linkage_matrix, k, criterion='maxclust')

# Plot the data and colour by cluster (using two features for visualisation)
plt.scatter(small_df['Customer_Age'], small_df['CLV'], c=clusters, cmap='viridis')
plt.title('Hierarchical clustering of CLV and Customer_Age')
plt.xlabel('Customer_Age')
plt.ylabel('CLV')
plt.xlim(small_df['Customer_Age'].min(), small_df['Customer_Age'].max())
plt.ylim(small_df['CLV'].min(), small_df['CLV'].max())
plt.colorbar(label='Cluster')
plt.show()

# Plot the dendrogram
plt.figure(figsize=(25, 10))
dendrogram(linkage_matrix)
plt.tick_params(labelbottom=False)
plt.title('Dendrogram')
plt.xlabel('Sample index or (Cluster size)')
plt.ylabel('Distance')
plt.show()

**Cluster Analysis**

* 1 & 2 (Purple): High CLV, older customers
* 3 & 4 (Blue/Green): Moderate CLV, varied ages
* 5 (Yellow): Low CLV, younger, less engaged
* Older = higher CLV, younger = lower CLV

**Dendrogram:** Initial split into two, then five clusters

In [ ]:
small_df = data_clean_scaled.sample(n=10000).reset_index(drop=True)

# Hierarchical clustering function
def hierarchical_clustering(data, method='ward'):
    dists = pdist(data, metric='euclidean')
    linkage_matrix = linkage(dists, method=method)
    return linkage_matrix

# Compute the linkage matrix
linkage_matrix = hierarchical_clustering(small_df)

# Number of clusters we want to extract
k = 5

# Retrieve the clusters
clusters = fcluster(linkage_matrix, k, criterion='maxclust')

# Plot the data and colour by cluster (using two features for visualisation)
plt.scatter(small_df['Recency'], small_df['CLV'], c=clusters, cmap='viridis')
plt.title('Hierarchical clustering of CLV and Recency')
plt.xlabel('Recency')
plt.ylabel('CLV')
plt.xlim(small_df['Recency'].min(), small_df['Recency'].max())
plt.ylim(small_df['CLV'].min(), small_df['CLV'].max())
plt.colorbar(label='Cluster')
plt.show()

# Plot the dendrogram
plt.figure(figsize=(25, 10))
dendrogram(linkage_matrix)
plt.tick_params(labelbottom=False)
plt.title('Dendrogram')
plt.xlabel('Sample index or (Cluster size)')
plt.ylabel('Distance')
plt.show()

**Cluster Analysis**

* 5 (Yellow): Low CLV, low recency; recent, valuable.
* 4 (Green): Moderate CLV, mixed recency; diverse engagement.
* 3 (Teal): Lower CLV, moderate recency.
* 2 (Blue): Lower CLV, high recency; less recent, low-value.
* 1 (Purple): High CLV, more or less recent.

**Dendrogram:** Clear initial split

In [ ]:
small_df = data_clean_scaled.sample(n=10000).reset_index(drop=True)

# Hierarchical clustering function
def hierarchical_clustering(data, method='ward'):
    dists = pdist(data, metric='euclidean')
    linkage_matrix = linkage(dists, method=method)
    return linkage_matrix

# Compute the linkage matrix
linkage_matrix = hierarchical_clustering(small_df)

# Number of clusters we want to extract
k = 5

# Retrieve the clusters
clusters = fcluster(linkage_matrix, k, criterion='maxclust')

# Plot the data and colour by cluster (using Recency and CLV for visualisation)
plt.scatter(small_df['Customer_Age'], small_df['CLV'], c=clusters, cmap='viridis')
plt.title('Hierarchical clustering of Customer_Age and CLV')
plt.xlabel('Customer_Age')
plt.ylabel('CLV')
plt.xlim(small_df['Customer_Age'].min(), small_df['Customer_Age'].max())
plt.ylim(small_df['CLV'].min(), small_df['CLV'].max())
plt.colorbar(label='Cluster')
plt.show()

# Plot the dendrogram
plt.figure(figsize=(25, 10))
dendrogram(linkage_matrix)
plt.title('Dendrogram')
plt.tick_params(labelbottom=False)
plt.xlabel('Sample index or (Cluster size)')
plt.ylabel('Distance')
plt.show()

**Cluster Analysis**

* 1 (Purple): High CLV, varied ages.
* 4 (Green): Moderate CLV, diverse ages.
* 3 (Teal): Lower CLV, middle-aged.
* 2 (Blue): Lower CLV, younger.
* 5 (Yellow): Lowest CLV, young, new/less engaged.

**Dendrogram Analysis:**
* Split at high distance, with further subdivisions.
* High merge heights indicate distinct clusters

### <font color="blue">**PART III:** Customer segments</font>

In [ ]:
# Creating a copy of the original customer data_clean_scaled data set
# to merge later customerID
data_clean_scaled.info()

data_clean_scaled = data_clean_scaled2.copy()

data_clean_scaled.info()

In [ ]:
# Check the columns of data_clean and data_aggr
print("Columns in data_clean:", data_clean.columns)
print("Columns in data_aggr:", data_aggr.columns)

# If 'Customer ID' is not in data_clean, merge it back from data_aggr
if 'Customer ID' not in data_clean.columns:
    data_clean = pd.merge(
        data_clean,
        data_aggr[['Customer ID']],
        left_index=True,  # Merge based on index
        right_index=True,
        how='left'
    )

# Now perform the merge operation for data_clean_Customer_ID
data_clean_Customer_ID = pd.merge(data_clean, data_aggr[['Customer ID']], on='Customer ID', how='left')

print(data_aggr_IQR_cal_cleaned.info())
print()
print()

# Merge 'data_clean' and 'data_aggr' to get 'Customer ID' for 'data_clean' rows
data_clean_Customer_ID = pd.merge(data_clean, data_aggr[['Customer ID']], on='Customer ID', how='left')

print(data_clean_Customer_ID.info())
print()
print()

# Impute missing values using the mean
imputer = SimpleImputer(strategy='mean')
data_clean_Customer_ID[['Average_Unit_Cost', 'Frequency', 'CLV', 'Recency', 'Customer_Age']] = imputer.fit_transform(
    data_clean_Customer_ID[['Average_Unit_Cost', 'Frequency', 'CLV', 'Recency', 'Customer_Age']])

# Perform KMeans clustering on the imputed data
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans_labels = kmeans.fit_predict(
    data_clean_Customer_ID[['Average_Unit_Cost', 'Frequency', 'CLV', 'Recency', 'Customer_Age']])

# Add the cluster labels to your original dataframe
data_clean_Customer_ID['Cluster'] = kmeans_labels

# Display the first few rows to see how clusters are associated with Customer ID
print(data_clean_Customer_ID[['Customer ID', 'Cluster']].head())
print()
print()
# For a more structured table view:
table = data_clean_Customer_ID[['Customer ID', 'Cluster']]
#print(table.to_string(index=False))

# Print summary statistics for each cluster
print(data_clean_Customer_ID.groupby('Cluster').agg({'Customer ID': 'count'}))
print()
print()

from IPython.display import display
display(table)

# Create boxplots to display the clusters with regard to frequency, recency, CLV, average unit cost, and customer age
for column in ['Frequency', 'Recency', 'CLV', 'Average_Unit_Cost', 'Customer_Age']:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='Cluster', y=column, data=data_clean_Customer_ID)
    plt.title(f'Boxplot of {column} by Cluster')
    plt.show()

**Analysis**

* Frequency: Cluster 3 highest engagement, Cluster 0 lowest with high variability.
* Recency: Cluster 1 most recent, Cluster 4 least recent.
* CLV: Cluster 3 highest value, Cluster 0 shows mix of values.
* Unit Cost: Cluster 0 & 4 variable spending, Cluster 3 lowest cost.
* Age: Cluster 0 oldest, Cluster 3 youngest with least variability.

### <font color="blue">**PART IV**: Dimensionality reduction</font>

In [ ]:
if 'Customer ID' not in data_aggr_IQR_cal_cleaned.columns:
    data_aggr_IQR_cal_cleaned = pd.merge(
        data_aggr_IQR_cal_cleaned,
        data_aggr[['Customer ID']],
        left_index=True,
        right_index=True,
        how='left'
    )

data_aggr_IQR_cal_cleaned = data_aggr_IQR_cal_cleaned.drop_duplicates(subset='Customer ID')
data_aggr = data_aggr.drop_duplicates(subset='Customer ID')

# Merge 'data_clean' and 'data_aggr' to get 'Customer ID' for 'data_clean' rows
data_clean_Customer_ID = pd.merge(data_clean, data_aggr[['Customer ID']], on='Customer ID', how='left')

In [ ]:
data_clean_Customer_ID.info()
print()
print()
print(data_clean_Customer_ID.describe())

In [ ]:
#  Standardising data_clean_Customer_ID

# columns to standardise
columns_to_standardise = ['Average_Unit_Cost', 'Frequency', 'CLV', 'Recency', 'Customer_Age']

# Initialise the StandardScaler
scaler = StandardScaler()

# Standardise the selected columns
data_clean_Customer_ID[columns_to_standardise] = scaler.fit_transform(data_clean_Customer_ID[columns_to_standardise])

# Check the first few rows to ensure standardisation is done
print(data_clean_Customer_ID.head())

In [ ]:
Missing_Values = data_clean_Customer_ID.isnull().sum()
print(Missing_Values)

In [ ]:
# 11. Perform dimension reduction with PCA and t-SNE to reduce the data to 2D.

selected_features = ['Average_Unit_Cost', 'Frequency', 'CLV', 'Recency', 'Customer_Age']

# Perform KMeans clustering on the imputed data
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans_labels = kmeans.fit_predict(
    data_clean_Customer_ID[selected_features])

# Add the cluster labels to your original dataframe
data_clean_Customer_ID['Cluster'] = kmeans_labels

In [ ]:
# Performing PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(data_clean_Customer_ID[selected_features])
data_clean_Customer_ID['PCA1'] = pca_result[:, 0]
data_clean_Customer_ID['PCA2'] = pca_result[:, 1]

In [ ]:
# Performing t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_result = tsne.fit_transform(data_clean_Customer_ID[selected_features])
data_clean_Customer_ID['tSNE1'] = tsne_result[:, 0]
data_clean_Customer_ID['tSNE2'] = tsne_result[:, 1]

In [ ]:
# Print PCA Loadings
pca_loadings = pd.DataFrame(pca.components_, columns=['Average_Unit_Cost', 'Frequency', 'CLV', 'Recency', 'Customer_Age'])
print("PCA Loadings (Components):")
print(pca_loadings)
print()
print()

In [ ]:
# 12. Create a 2D visualisation to display the clusters with different
# colours. Use the output from the PCA and t-SNE.

# Visualise PCA results
plt.figure(figsize=(16, 7))
sns.scatterplot(x='PCA1', y='PCA2', hue='Cluster', palette='viridis', data=data_clean_Customer_ID, legend='full')
plt.title('PCA of Customer Data')

In [ ]:
# The effectiveness of PCA in capturing the variance the data can be assessed
# by looking at the explained variance ratio

explained_variance_ratio = pca.explained_variance_ratio_
print("Explained Variance Ratio:", explained_variance_ratio)

In [ ]:
# Print t-SNE Coordinates
tsne_df = pd.DataFrame(tsne_result, columns=['tSNE1', 'tSNE2'])
print("t-SNE Coordinates:")
print(tsne_df.head())

In [ ]:
# Visualise t-SNE results
plt.figure(figsize=(10, 7))
sns.scatterplot(x='tSNE1', y='tSNE2', hue='Cluster', palette='viridis', data=data_clean_Customer_ID, legend='full')
plt.title('t-SNE of Customer Data')
plt.show()

In [ ]:
# n_clusters = 4

# 11. Perform dimension reduction with PCA and t-SNE to reduce the data to 2D.

selected_features = ['Average_Unit_Cost', 'Frequency', 'CLV', 'Recency', 'Customer_Age']

# Perform KMeans clustering on the imputed data
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans_labels = kmeans.fit_predict(
    data_clean_Customer_ID[selected_features])

# Add the cluster labels to your original dataframe
data_clean_Customer_ID['Cluster'] = kmeans_labels

# Performing PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(data_clean_Customer_ID[selected_features])
data_clean_Customer_ID['PCA1'] = pca_result[:, 0]
data_clean_Customer_ID['PCA2'] = pca_result[:, 1]

# Performing t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_result = tsne.fit_transform(data_clean_Customer_ID[selected_features])
data_clean_Customer_ID['tSNE1'] = tsne_result[:, 0]
data_clean_Customer_ID['tSNE2'] = tsne_result[:, 1]

# 12. Create a 2D visualisation to display the clusters with different
# colours. Use the output from the PCA and t-SNE.

# Print PCA Loadings
pca_loadings = pd.DataFrame(pca.components_, columns=['Average_Unit_Cost', 'Frequency', 'CLV', 'Recency', 'Customer_Age'])
print("PCA Loadings (Components):")
print(pca_loadings)
print()
print()

# Visualise PCA results
plt.figure(figsize=(16, 7))
sns.scatterplot(x='PCA1', y='PCA2', hue='Cluster', palette='viridis', data=data_clean_Customer_ID, legend='full')
plt.title('PCA of Customer Data')

# Print t-SNE Coordinates
tsne_df = pd.DataFrame(tsne_result, columns=['tSNE1', 'tSNE2'])
print("t-SNE Coordinates:")
print(tsne_df.head())

# Visualise t-SNE results
plt.figure(figsize=(10, 7))
sns.scatterplot(x='tSNE1', y='tSNE2', hue='Cluster', palette='viridis', data=data_clean_Customer_ID, legend='full')
plt.title('t-SNE of Customer Data')
plt.show()

In [ ]:
# The effectiveness of PCA in capturing the variance the data can be assessed
# by looking at the explained variance ratio

explained_variance_ratio = pca.explained_variance_ratio_
print("Explained Variance Ratio:", explained_variance_ratio)